# PII Guardrail Model Training

Fine-tune Llama 3.2-1B for PII detection using QLoRA with Unsloth.

**Target Hardware**: Google Colab T4 GPU (16GB VRAM)

**Model**: `unsloth/Llama-3.2-1B-Instruct`

**Technique**: QLoRA (4-bit quantization with LoRA adapters)

## Features
- Detects Indian PII: Aadhaar, PAN
- Detects General PII: Email, Phone, Names, SSN, Credit Cards
- Outputs JSON with entity types, positions, confidence, and reasons


In [ ]:
# Cell 1: Install Unsloth and dependencies
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers scikit-learn


In [ ]:
# Cell 2: Mount Google Drive and setup configuration
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import torch
from datetime import datetime

# Configuration
CONFIG = {
    "model_name": "unsloth/Llama-3.2-1B-Instruct",
    "max_seq_length": 512,
    "load_in_4bit": True,
    "lora_rank": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "num_epochs": 3,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.1,
    "output_dir": "/content/drive/MyDrive/pii-guardrail-model",
}

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Cell 3: Load Model with QLoRA (4-bit quantization)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,  # Auto-detect
)

print(f"Model loaded: {CONFIG['model_name']}")
print(f"Model parameters: {model.num_parameters():,}")


In [ ]:
# Cell 4: Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_rank"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")


In [ ]:
# Cell 5: Generate Synthetic Training Data
import random
import string

def generate_aadhaar():
    first = str(random.randint(2, 9))
    rest = ''.join(random.choices(string.digits, k=11))
    num = first + rest
    return f"{num[:4]} {num[4:8]} {num[8:12]}"

def generate_pan():
    letters = ''.join(random.choices(string.ascii_uppercase, k=5))
    digits = ''.join(random.choices(string.digits, k=4))
    check = random.choice(string.ascii_uppercase)
    return f"{letters}{digits}{check}"

def generate_email():
    names = ['rahul', 'priya', 'amit', 'neha', 'john', 'sarah', 'darshan', 'krishna']
    domains = ['gmail.com', 'yahoo.com', 'outlook.com', 'company.in']
    return f"{random.choice(names)}{random.randint(1,99)}@{random.choice(domains)}"

def generate_phone():
    first = random.choice(['6', '7', '8', '9'])
    rest = ''.join(random.choices(string.digits, k=9))
    return f"+91 {first}{rest[:4]} {rest[4:]}"

def generate_name():
    first = random.choice(['Rahul', 'Priya', 'Amit', 'Neha', 'Darshan', 'Krishna', 'Lakshmi'])
    last = random.choice(['Sharma', 'Patel', 'Singh', 'Kumar', 'Gupta', 'Reddy'])
    return f"{first} {last}"

def generate_ssn():
    area = random.randint(100, 899)
    group = random.randint(10, 99)
    serial = random.randint(1000, 9999)
    return f"{area}-{group}-{serial}"

# Templates with context words (Presidio-aligned)
TEMPLATES = [
    # Aadhaar templates
    ("My Aadhaar number is {value}.", "IN_AADHAAR", generate_aadhaar),
    ("Aadhaar: {value}", "IN_AADHAAR", generate_aadhaar),
    ("UIDAI Aadhaar: {value} for verification.", "IN_AADHAAR", generate_aadhaar),
    ("My unique identification number is {value}.", "IN_AADHAAR", generate_aadhaar),
    ("UID: {value} for KYC.", "IN_AADHAAR", generate_aadhaar),
    # PAN templates
    ("My PAN card number is {value}.", "IN_PAN", generate_pan),
    ("PAN: {value}", "IN_PAN", generate_pan),
    ("For tax purposes, my PAN is {value}.", "IN_PAN", generate_pan),
    ("Permanent Account Number: {value}", "IN_PAN", generate_pan),
    ("Income tax PAN: {value}", "IN_PAN", generate_pan),
    # Email templates
    ("My email address is {value}.", "EMAIL_ADDRESS", generate_email),
    ("Contact me at {value}.", "EMAIL_ADDRESS", generate_email),
    ("Email: {value}", "EMAIL_ADDRESS", generate_email),
    ("Send documents to {value}.", "EMAIL_ADDRESS", generate_email),
    # Phone templates
    ("My phone number is {value}.", "PHONE_NUMBER", generate_phone),
    ("Call me at {value}.", "PHONE_NUMBER", generate_phone),
    ("Mobile: {value}", "PHONE_NUMBER", generate_phone),
    ("Contact number: {value}", "PHONE_NUMBER", generate_phone),
    # Person templates
    ("My name is {value}.", "PERSON", generate_name),
    ("I am {value}.", "PERSON", generate_name),
    ("This is {value} speaking.", "PERSON", generate_name),
    # SSN templates
    ("My SSN is {value}.", "US_SSN", generate_ssn),
    ("Social Security Number: {value}", "US_SSN", generate_ssn),
]

NEGATIVE_TEMPLATES = [
    "The weather today is sunny with a high of 25 degrees.",
    "Please submit the report by Friday.",
    "The meeting is scheduled for 3 PM tomorrow.",
    "Thank you for your patience.",
    "The project deadline has been extended.",
    "Please review the attached document.",
    "The system will be under maintenance tonight.",
    "Your request has been processed successfully.",
    "The quarterly results exceeded expectations.",
    "Please confirm your attendance.",
]

SYSTEM_PROMPT = """You are a PII detection model. Analyze text and identify PII entities with their exact positions.

Output JSON with: flagged (bool), entities (array with type, value, start, end), confidence (0-1), reason (string).

Entity types: IN_AADHAAR, IN_PAN, EMAIL_ADDRESS, PHONE_NUMBER, PERSON, US_SSN, CREDIT_CARD"""

def generate_sample():
    if random.random() < 0.2:  # 20% negative samples
        text = random.choice(NEGATIVE_TEMPLATES)
        output = {
            "flagged": False,
            "entities": [],
            "confidence": 1.0,
            "reason": "No PII detected"
        }
    else:
        template, entity_type, generator = random.choice(TEMPLATES)
        value = generator()
        text = template.replace("{value}", value)
        start = text.find(value)
        end = start + len(value)
        
        readable_type = entity_type.replace("_", " ").replace("IN ", "").title()
        output = {
            "flagged": True,
            "entities": [{"type": entity_type, "value": value, "start": start, "end": end}],
            "confidence": 1.0,
            "reason": f"Detected 1 {readable_type}"
        }
    
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f'Detect PII in: "{text}"'},
            {"role": "assistant", "content": json.dumps(output, indent=2)}
        ]
    }

# Generate dataset
NUM_SAMPLES = 5000
train_data = [generate_sample() for _ in range(int(NUM_SAMPLES * 0.8))]
eval_data = [generate_sample() for _ in range(int(NUM_SAMPLES * 0.2))]

print(f"Generated {len(train_data)} training samples")
print(f"Generated {len(eval_data)} evaluation samples")

# Save to files
with open('train.jsonl', 'w') as f:
    for item in train_data:
        f.write(json.dumps(item) + '\n')

with open('eval.jsonl', 'w') as f:
    for item in eval_data:
        f.write(json.dumps(item) + '\n')

print("\nSample:")
print(json.dumps(train_data[0], indent=2))


In [ ]:
# Cell 6: Load and format datasets
from datasets import load_dataset

train_dataset = load_dataset('json', data_files='train.jsonl', split='train')
eval_dataset = load_dataset('json', data_files='eval.jsonl', split='train')

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Eval dataset: {len(eval_dataset)} samples")

def format_chat(example):
    """Format conversations into the Llama chat template."""
    formatted = tokenizer.apply_chat_template(
        example['conversations'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted}

train_dataset = train_dataset.map(format_chat)
eval_dataset = eval_dataset.map(format_chat)

print("\nFormatted sample preview:")
print(train_dataset[0]['text'][:500])


In [ ]:
# Cell 7: Setup Training
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    args=training_args,
)

print("Trainer initialized!")
print(f"Training for {CONFIG['num_epochs']} epochs")
print(f"Effective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")


In [ ]:
# Cell 8: Train the model
print(f"Starting training at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

trainer_stats = trainer.train()

print(f"\nTraining completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Training loss: {trainer_stats.training_loss:.4f}")


In [ ]:
# Cell 9: Evaluation - Calculate Precision, Recall, F1 per entity type
from sklearn.metrics import precision_recall_fscore_support
from collections import defaultdict

def extract_entities_from_output(output_text):
    """Extract entities from model output JSON."""
    try:
        start = output_text.find('{')
        end = output_text.rfind('}') + 1
        if start >= 0 and end > start:
            data = json.loads(output_text[start:end])
            return data.get('entities', []), data.get('flagged', False)
    except:
        pass
    return [], False

def evaluate_model(model, tokenizer, eval_samples, max_samples=100):
    """Evaluate model on eval set."""
    FastLanguageModel.for_inference(model)
    
    results = {
        'true_flagged': [],
        'pred_flagged': [],
        'true_entities': [],
        'pred_entities': [],
    }
    
    for i, sample in enumerate(eval_samples[:max_samples]):
        if i % 20 == 0:
            print(f"Evaluating sample {i+1}/{max_samples}...")
        
        conversations = sample['conversations']
        ground_truth = json.loads(conversations[2]['content'])
        
        messages = conversations[:2]
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        output_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        pred_entities, pred_flagged = extract_entities_from_output(output_text)
        
        results['true_flagged'].append(ground_truth['flagged'])
        results['pred_flagged'].append(pred_flagged)
        results['true_entities'].append(ground_truth['entities'])
        results['pred_entities'].append(pred_entities)
    
    return results

print("Running evaluation...")
eval_results = evaluate_model(model, tokenizer, eval_data, max_samples=100)


In [ ]:
# Cell 10: Print Precision-Recall Report
precision, recall, f1, _ = precision_recall_fscore_support(
    eval_results['true_flagged'],
    eval_results['pred_flagged'],
    average='binary'
)

print("=" * 60)
print("PII DETECTION METRICS - PRECISION/RECALL REPORT")
print("=" * 60)
print(f"\nOverall Flagged Detection:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")

# Per-entity-type metrics
entity_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

for true_ents, pred_ents in zip(eval_results['true_entities'], eval_results['pred_entities']):
    true_set = {(e['type'], e['value']) for e in true_ents}
    pred_set = {(e['type'], e['value']) for e in pred_ents}
    
    for etype, value in true_set:
        if (etype, value) in pred_set:
            entity_metrics[etype]['tp'] += 1
        else:
            entity_metrics[etype]['fn'] += 1
    
    for etype, value in pred_set:
        if (etype, value) not in true_set:
            entity_metrics[etype]['fp'] += 1

print(f"\nPer-Entity-Type Metrics:")
print("-" * 60)
print(f"{'Entity Type':<20} {'Precision':>12} {'Recall':>12} {'F1':>12}")
print("-" * 60)

for etype in sorted(entity_metrics.keys()):
    counts = entity_metrics[etype]
    tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f"{etype:<20} {prec:>12.4f} {rec:>12.4f} {f1_score:>12.4f}")

print("=" * 60)


In [ ]:
# Cell 11: Save LoRA adapters
lora_path = f"{CONFIG['output_dir']}/lora-adapters"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

print(f"LoRA adapters saved to: {lora_path}")

# Save training config and metrics
metrics = {
    "config": CONFIG,
    "training_loss": trainer_stats.training_loss,
    "evaluation": {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    },
    "entity_metrics": {k: dict(v) for k, v in entity_metrics.items()},
    "timestamp": datetime.now().isoformat(),
}

metrics_path = f"{CONFIG['output_dir']}/training_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics saved to: {metrics_path}")


In [ ]:
# Cell 12: Test Inference with sample inputs
FastLanguageModel.for_inference(model)

test_texts = [
    "My Aadhaar number is 2345 6789 0123.",
    "Contact me at rahul.sharma@gmail.com or call +91 98765 43210.",
    "My PAN card number is ABCPD1234E for tax filing.",
    "The weather today is sunny with a high of 25 degrees.",
    "User Darshan Krishna with Aadhaar 9876 5432 1098 and PAN BNZPM2501F registered.",
]

print("=" * 70)
print("TEST INFERENCE RESULTS")
print("=" * 70)

for text in test_texts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Detect PII in: "{text}"'}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    output_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    print(f"\nInput: {text}")
    print(f"Output:\n{output_text}")
    print("-" * 70)


In [ ]:
# Cell 13: Export for deployment - zip LoRA adapters
import shutil

zip_path = "/content/pii-guardrail-lora"
shutil.make_archive(zip_path, 'zip', lora_path)

print(f"Created: {zip_path}.zip")
print("\nTo download, uncomment and run:")
print("# from google.colab import files")
print(f"# files.download('{zip_path}.zip')")
